Newton을 이용해 단일 무부하 서보 모터의 1-DOF 디지털 트윈 모델을 구성한다.

현재 이 파일은

- Newton에 1-DOF 회전 모터 모델을 구성
- `MotorParams`를 통해 제어기 및 모터 물리 파라미터 정의
- Revolute Joint 기반의 단일 모터 모델 생성
- Command Position을 입력받아 모터의 동작을 계산하는 `simulate_motor()` 구현
- Position / Velocity / Torque를 출력할 수 있는 시뮬레이션 구조 구성

정도로 구성되어 있다.

이 파일에서는 특정 시험 지령을 생성하거나 직접 시험을 수행하지 않고,
다른 Notebook에서 공통으로 사용할 **Newton 단일 모터 모델 자체를 정의하는 것**을 목적으로 한다.

## Import
현재는 단일 무부하 모터에 대해 디지털 트윈을 구성하기에, franka의 예시처럼 `newton.examples`, `newton.utils`, `newton.ik` 과 같은 기능은 필요하지 않다.

In [1]:
from dataclasses import dataclass

import numpy as np
import warp as wp
import newton

wp.config.log_level = wp.LOG_WARNING

## Motor Parameter Setup

**MotorParams**는 단일 모터 디지털 트윈에서 사용할 제어 및 물리 파라미터를 하나로 묶어 관리하기 위한 class이다.   
현재 설정된 초기값은 실제 모터의 확정값이 아닌 추후 **WMX3 실측 데이터와 비교하여 조정할 피팅 대상 값**이다. 

제어기 관련 파라미터 :
- `kp`, `ki`, `kd`
- `integral_max`
- `effort_limit`
- `delay_steps`

모터 물리 특성 관련 파라미터 :
- `inertia`
- `viscous`
- `coulomb`

In [2]:
@dataclass
class MotorParams:
    # ---- 피팅 대상 파라미터 (초기값은 자리표시자) ----
    #Drive / Controller
    kp: float = 8.0            # 위치 (오차에 비례해 토크를 생성하는) 비례 게인
    ki: float = 0.0            # (누적된 위치 오차를 보정하는) 적분 게인
    kd: float = 0.6            # (모터의 속도에 따른 감쇠 역할을 하는) 미분(속도) 게인
    integral_max: float = 5.0  # 적분값이 과도하게 누적되는 것을 제한하는 anti-windup 한계

    # Motor/ Drive Limit
    effort_limit: float = 5.0  # 토크 포화 [N·m] : 모터가 출력 가능한 토크의 최대값

    # Mechanical Plant
    inertia: float = 0.02      # 유효(반사) 관성 J [kg·m^2]
    viscous: float = 0.01      # 점성 마찰 b [N·m·s/rad] : 속도에 비례해 발생
    coulomb: float = 0.0       # 쿨롱 마찰 [N·m] : 운동 방향의 반대 방향으로 작용

    # Communication / Computation Delay
    delay_steps: int = 0       # Command Position이 실제 제어 계산에 반영되기까지의 지연 사이클

## Build a Single Motor Model

articulation은 로보틱스/물리 시뮬레이션에서 여러 rigid body(강체)가 joint(관절)로 연결된 하나의 관절 시스템을 뜻함.  

회전축 하나만 있는 articulation으로 만들기.  
=> 고정된 베이스와 회전하는 body를 하나의 revolute joint로 연결해서 1-DOF 관절 시스템을 만든다.


In [3]:
def build_motor_model(p: MotorParams, device="cpu"):

    # 1. Scene : 외부 부하/중력 토크를 제외한 무부하 축 응답을 보기 위해 gravity=0
    builder = newton.ModelBuilder(gravity=0.0)

    # 2. Rotor inertia
    J = max(float(p.inertia), 1.0e-6)

    inertia_tensor = wp.mat33(
        J,   0.0, 0.0,
        0.0, J,   0.0,
        0.0, 0.0, J,
    )
    rotor = builder.add_link(
        mass=1.0e-3,
        inertia=inertia_tensor,
        lock_inertia=True,
        label="motor_rotor",
    )

    # 3. Viewer용 형상
    # as_site=True:
    # 시각화만 하고 질량/관성/충돌에는 영향을 주지 않음
    builder.add_shape_cylinder(
        rotor,
        radius=0.05,
        half_height=0.015,
        as_site=True,
        label="rotor_visual",
    )
    # 회전 여부가 잘 보이도록 한쪽에 marker 추가
    builder.add_shape_box(
        rotor,
        xform=wp.transform(
            wp.vec3(0.04, 0.0, 0.025),
            wp.quat_identity(),
        ),
        hx=0.035,
        hy=0.006,
        hz=0.006,
        as_site=True,
        label="rotor_marker",
    )

    # 4. 1-DOF Revolute Joint
    joint = builder.add_joint_revolute(
        parent=-1,                  # world
        child=rotor,
        axis=wp.vec3(0.0, 0.0, 1.0),

        # Newton 내장 joint target controller 사용 X
        target_ke=0.0,
        target_kd=0.0,

        # 숨은 damping/관성 제거
        damping=0.0,
        armature=0.0,

        # Featherstone에서는 아래 기능을 사용하지 않을 것이므로
        # 사실상 비활성화
        friction=0.0,
        effort_limit=1.0e9,
        velocity_limit=1.0e9,

        label="motor_joint",
    )

    # 5. Articulation
    builder.add_articulation(
        [joint],
        label="single_servo_motor",
    )


    # 6. Simulation Model
    model = builder.finalize(device=device)

    return model


## 실제 Servo Drive + Motor Simulation

`simulate_motor()`는 CSP Command Position을 입력으로 받아, 지연·PID 제어·토크 포화·점성/쿨롱 마찰을 적용한 뒤 Newton으로 단일 서보 모터의 운동을 시뮬레이션하는 함수이다.

Feedback Position / Velocity / Torque, Net Torque, Position Error를 기록하고 필요하면 Viewer로 시각화한다.

CSP Command Position [rad] : 시간에 따라 들어오는 위치 지령들  
`cmd_pos` : (N,) ndarray -> 길이가 N인 1차원 Numpy 배열

WMX control / communication cycle [s] : 한 번의 제어 계산 사이 시간 간격  
`dt` : float 

`p` : MotorParams -> 내가 정의해 놓은 모터 parameter가 들어감  

함수 결과는 dictionary 형태로 돌려준다.  

In [4]:
# simulate_motor() 라는 함수를 정의
def simulate_motor(
    cmd_pos,
    dt,
    p: MotorParams,
    device="cpu",
    use_viewer=True,
    viewer_fps=60,
    initial_position=0.0,
    initial_velocity=0.0,
):

    cmd_pos = np.asarray(cmd_pos, dtype=np.float32) # 입력받은 cmd_pos를 numpy 배열로 변환
    N = len(cmd_pos) # 지령이 총 몇 개인지 저장



    # Newton Model 생성 --------------------------------------
    model = build_motor_model(p, device=device)

    # Newton 모터축의 초기 위치와 초기 속도 설정
    model.joint_q.assign(
        np.array(
            [initial_position],
            dtype=np.float32,
        )
    )

    model.joint_qd.assign(
        np.array(
            [initial_velocity],
            dtype=np.float32,
        )
    )

    state_0 = model.state()
    state_1 = model.state()
    control = model.control()


    # 초기 joint coordinate → body pose 계산
    newton.eval_fk(   # 순기구학
        model,
        model.joint_q,  # 관절 좌표
        model.joint_qd, # 관절 속도
        state_0,        # 위의 두 값에 맞는 실제 body의 위치,회전,속도를 계산해서 state에 넣는다
    )

    newton.eval_fk(
        model,
        model.joint_q,
        model.joint_qd,
        state_1,
    )

    # Featherstone solver --------------------------------------
    # 현재 모터 상태와 입력 토크를 가지고 다음 위치와 속도를 계산
    # 중요 : 기본 angular_damping=0.05를 끈다.
    # 그래야 우리가 정의한 viscous만 damping으로 작용해서 식별 파라미터가 섞이지 않는다.
    solver = newton.solvers.SolverFeatherstone(
        model,
        angular_damping=0.0,
    )

    # Logging buffer --------------------------------------
    time_log = np.arange(N, dtype=np.float64) * dt   # 시간 

    fb_pos = np.zeros(N, dtype=np.float64)           # 모터 위치 저장용
    fb_vel = np.zeros(N, dtype=np.float64)           # 모터 속도 저장용

    # WMX Feedback Torque와 우선 비교할 모터 발생 토크
    fb_trq = np.zeros(N, dtype=np.float64)

    # 모터 토크에서 점성/쿨롱 마찰 토크를 차감한 plant 입력 net torque
    net_trq = np.zeros(N, dtype=np.float64)

    err_log = np.zeros(N, dtype=np.float64)           # 위치 오차 저장용


    # Controller state --------------------------------------
    integral = 0.0                # PID 중 I항의 누적값
    joint_force = np.zeros(       # Newton의 joint애 놓을 force/torque 배열
        model.joint_dof_count,
        dtype=np.float32,
    )

    # Viewer --------------------------------------
    viewer = None

    if use_viewer:
        viewer = newton.viewer.ViewerViser(
            label="Single Servo Motor Digital Twin",
            record_to_viser="single_motor.viser",   # ← 이것만 추가
            plot_history_size=500,
        )

        viewer.set_model(model)

        try:
            viewer.set_camera(
                wp.vec3(0.25, -0.30, 0.18),
                -20,
                130,
            )
        except Exception:
            pass

    # 약 60 FPS만 기록
    render_stride = max(
        1,
        int(round(1.0 / (viewer_fps * dt)))
    )

    # Simulation Loop -------------------------------------- (중요)
    for k in range(N):

        # A. 현재 피드백 -> 현재 모터 상태를 가져옴
        q = float(state_0.joint_q.numpy()[0])       # 현재위치, 현재 모터축 각도
        qd = float(state_0.joint_qd.numpy()[0])     # 현재속도, 현재 각속도
        
        # 현재 timestep의 Feedback 기록
        fb_pos[k] = q
        fb_vel[k] = qd

        # B. Command Delay
        delayed_index = max(0, k - p.delay_steps)
        cmd = float(cmd_pos[delayed_index])

        # C. Position error
        error = cmd - q              # 목표와 현재 위치의 차이

        # D. Integral + anti-windup
        integral += error * dt       # 누적 오차

        integral = float(            # 누적 오차가 무한히 커지는 것을 막음
            np.clip(
                integral,
                -p.integral_max,
                p.integral_max,
            )
        )

        # E. PID
        # τ = Kp e + Ki ∫e dt - Kd qdot  :  컨트롤러가 토크를 계산하는 핵심 식 
        tau_motor = (
            p.kp * error          # kp : 위치 오차가 크면 강하게 밀어라 
            + p.ki * integral     # ki : 계속 남는 오차를 없애라
            - p.kd * qd           # kd : 너무 빠르게 움직이면 제동을 걸어라
        )

        # F. Motor torque saturation
        tau_motor = float(        # 모터가 낼 수 있는 최대 토크를 제한
            np.clip(
                tau_motor,
                -p.effort_limit,
                p.effort_limit,
            )
        )

        # G. Mechanical friction
        tau_viscous = p.viscous * qd    # 점성 마찰 : 속도가 빠를수록 커진다

        tau_coulomb = (                 # 쿨롱 마찰 : 속도의 크기 보다는 움직이는 
            p.coulomb * np.sign(qd)                # 방향에 따라 반대 방향으로 일정한 마찰
        )

        # H. Net torque
        tau_net = (         # 실제 Newton에 들어갈 최종 토크
            tau_motor
            - tau_viscous
            - tau_coulomb
        )

        # I. Newton Control.joint_f  : Newton control에 토크 넣기
        joint_force[0] = tau_net              # 1-DOF joint의 토크 입력값에 tau_net을 넣음
        control.joint_f.assign(joint_force)   # 그 값을 Newton Control 객체에 전달

        # J. Physics integration
        state_0.clear_forces()

        solver.step(    # solver가 값들을 가지고 계산하고 그 밧을 state_1에 저장
            state_0,
            state_1,
            control,
            None,
            dt,
        )
        state_0, state_1 = state_1, state_0  # 계산된 state_1을 현재상태로 바꾸고 state_1은 다시 비워두기

        # K. Solver 이후의 Feedback 기록
        q_next = float(state_0.joint_q.numpy()[0])     # solver가 계산한 새 위치를 가져옴
        qd_next = float(state_0.joint_qd.numpy()[0])   # solver가 계산한 새 속도를 가져옴
        

        # 우선 실제 Drive Feedback Torque와 비교할 대상
        fb_trq[k] = tau_motor

        # 마찰을 반영한 후 Newton 물리축에 적용되는 최종 토크
        net_trq[k] = tau_net

        err_log[k] = error

        # L. Viewer
        if viewer is not None and k % render_stride == 0:

            sim_time = k * dt

            viewer.begin_frame(sim_time)

            viewer.log_state(state_0)

            # Viewer 안에서도 신호 확인 가능
            viewer.log_scalar(
                "Motor/Command Position",
                float(cmd_pos[k]),
            )

            viewer.log_scalar(
                "Motor/Feedback Position",
                q_next,
            )

            viewer.log_scalar(
                "Motor/Feedback Velocity",
                qd_next,
            )

            viewer.log_scalar(
                "Motor/Motor Torque",
                tau_motor,
            )

            viewer.end_frame()

    return {
        "time": time_log,
        "command_position": cmd_pos,
        "feedback_position": fb_pos,
        "feedback_velocity": fb_vel,
        "feedback_torque": fb_trq,
        "net_torque": net_trq,
        "position_error": err_log,
        "viewer": viewer,
    }